# Besoin Client 1 — Visualisation sur carte
**Projet Big Data / IA / Web — Partie IA — FISE3 2026**

## Objectif
Ce notebook répond précisément au Besoin Client 1 du cahier des charges :
1. Préparer, nettoyer et auditer géographiquement les données IRVE.
2. Analyser statistiquement la distribution des implantations et des puissances nominales.
3. Créer une carte interactive regroupant les points de recharge par type d'implantation sans saturer le navigateur.
4. Générer une carte de chaleur (Heatmap) globale et une vue filtrée sur la recharge rapide pour détecter les densités de déploiement et les anomalies.

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import folium
from folium.plugins import HeatMap, MarkerCluster
import warnings
warnings.filterwarnings('ignore')

# 1. Chargement du jeu de données complet issu de la partie Big Data
print("Chargement de la base de données...")
df = pd.read_csv('IRVE_clean_FINAL.csv')

print(f"Nombre de lignes initiales : {df.shape[0]}")
print(f"Nombre de colonnes initiales : {df.shape[1]}")
df.head(3)

Chargement de la base de données...
Nombre de lignes initiales : 70394
Nombre de colonnes initiales : 38


,nom_amenageur,contact_amenageur,nom_operateur,contact_operateur,telephone_operateur,nom_enseigne,id_station_itinerance,nom_station,implantation_station,adresse_station,...,station_deux_roues,raccordement,date_mise_en_service,date_maj,cable_t2_attache,consolidated_code_postal,consolidated_commune,lon,lat,tarif_kwh_clean
0,SOLVEO ENERGIES,bornes@solveo-energies.com,SOLVEO ENERGIES,bornes@solveo-energies.com,05 32 98 01 58,SOLVEO ENERGIES,DKMONE3785709,Montestruc,Parking public,Rue du 19 mars 1962,...,0,Direct,2024-11-28,2025-01-31,0,32390,Montestruc-sur-Gers,0.628264,43.792972,0.59
1,SOLVEO ENERGIES,bornes@solveo-energies.com,SOLVEO ENERGIES,bornes@solveo-energies.com,05 32 98 01 58,SOLVEO ENERGIES,DKMONE3785711,Montestruc,Parking public,Rue du 19 mars 1962,...,0,Direct,2024-11-28,2025-01-31,0,32390,Montestruc-sur-Gers,0.628264,43.792972,0.59
2,SOLVEO ENERGIES,bornes@solveo-energies.com,SOLVEO ENERGIES,bornes@solveo-energies.com,05 32 98 01 58,SOLVEO ENERGIES,DKMONE3785713,Montestruc,Parking public,Rue du 19 mars 1962,...,0,Direct,2024-11-28,2025-01-31,0,32390,Montestruc-sur-Gers,0.628264,43.792972,0.38


## Sélection et Nettoyage :
Pour que notre carte charge rapidement, on ne garde que les colonnes nécessaires (coordonnées et type d'implantation). Ensuite, on vérifie que nos points se trouvent bien en France métropolitaine pour éviter les valeurs aberrantes.

In [12]:
# Sélection des colonnes utiles pour ce besoin spécifique
colonnes_interet = ['nom_station', 'implantation_station', 'puissance_nominale',
                    'consolidated_commune', 'lat', 'lon']

df_bc1 = df[colonnes_interet].copy() # Création d'une copie du DataFrame avec les colonnes sélectionnées

print('Colonnes sélectionnées et types associés :')
print(df_bc1.dtypes) # affichage des types de données des colonnes sélectionnées
print()
df_bc1.head(5) # affichage des premières lignes du DataFrame df_bc1

Colonnes sélectionnées et types associés :
nom_station                 str
implantation_station        str
puissance_nominale      float64
consolidated_commune        str
lat                     float64
lon                     float64
dtype: object



,nom_station,implantation_station,puissance_nominale,consolidated_commune,lat,lon
0,Montestruc,Parking public,150.0,Montestruc-sur-Gers,43.792972,0.628264
1,Montestruc,Parking public,150.0,Montestruc-sur-Gers,43.792972,0.628264
2,Montestruc,Parking public,11.0,Montestruc-sur-Gers,43.792972,0.628264
3,Montestruc,Parking public,11.0,Montestruc-sur-Gers,43.792972,0.628264
4,Tommy's,Parking privé à usage public,11.0,Labège,43.539773,1.512599


### 2.2 Nettoyage géométrique
Nous validons l'intégrité des coordonnées GPS en éliminant les valeurs manquantes éventuelles, puis nous effectuons un filtrage géographique pour isoler la France métropolitaine et détecter les anomalies de saisie (points hors du territoire).

In [9]:
# Vérification et suppression des valeurs manquantes sur les coordonnées GPS
print('=== Valeurs manquantes ===') 
print(df_bc1.isnull().sum()) # affichage du nombre de valeurs manquantes par colonne
print() # ajout d'une ligne vide pour la lisibilité

avant = len(df_bc1) # sauvegarde du nombre de lignes avant suppression des valeurs manquantes
df_bc1 = df_bc1.dropna(subset=['lat', 'lon']) # suppression des lignes avec des valeurs manquantes sur les colonnes 'lat' et 'lon'
apres = len(df_bc1) # sauvegarde du nombre de lignes après suppression des valeurs manquantes
print(f"Lignes supprimées (lat/lon manquants) : {avant - apres}")

# Détection des anomalies géographiques (France métropolitaine grossièrement entre lat [41, 51.5] et lon [-5, 10])
print('\n=== Vérification des frontières géographiques ===')
print(f"Latitude  → min : {df_bc1['lat'].min():.4f} | max : {df_bc1['lat'].max():.4f}")
print(f"Longitude → min : {df_bc1['lon'].min():.4f} | max : {df_bc1['lon'].max():.4f}")

anomalies = df_bc1[(df_bc1['lat'] < 41) | (df_bc1['lat'] > 51.5) |
                   (df_bc1['lon'] < -5) | (df_bc1['lon'] > 10)]

print(f"Nombre de bornes détectées hors zone (anomalies géographiques) : {len(anomalies)}")

=== Valeurs manquantes ===
nom_station             0
implantation_station    0
puissance_nominale      0
consolidated_commune    0
lat                     0
lon                     0
dtype: int64

Lignes supprimées (lat/lon manquants) : 0

=== Vérification des frontières géographiques ===
Latitude  → min : 41.4925 | max : 51.0631
Longitude → min : -4.7505 | max : 9.5478
Nombre de bornes détectées hors zone (anomalies géographiques) : 0


### 2.3 Définition de la charte graphique par type d'implantation
Nous associons à chaque catégorie unique de la variable `implantation_station` une couleur hexadécimale stricte qui servira de fil conducteur visuel pour nos graphiques et nos calques de cartes.

In [13]:
# Recensement des catégories réelles de notre base filtrée
types_implantation = df_bc1['implantation_station'].value_counts() # affichage du nombre de stations par type d'implantation
print("=== Catégories d'implantation recensées ===")
print(types_implantation)

# Charte graphique Hexadécimale uniforme
COULEURS = {
    'Voirie'                                : '#2196F3',  # Bleu
    'Parking privé à usage public'          : '#4CAF50',  # Vert
    'Parking public'                        : '#FF9800',  # Orange
    'Station dédiée à la recharge rapide'   : '#F44336',  # Rouge
    'Parking privé réservé à la clientèle'  : '#9C27B0',  # Violet
}

# Ajout d'une colonne technique pour l'injection directe dans les graphiques
df_bc1['couleur_hex'] = df_bc1['implantation_station'].map(COULEURS).fillna('#757575')

=== Catégories d'implantation recensées ===
implantation_station
Voirie                                  27253
Parking privé à usage public            20891
Parking public                          13333
Station dédiée à la recharge rapide      8128
Parking privé réservé à la clientèle      789
Name: count, dtype: int64


### 3. Visualisation Statistique de la Distribution

Avant de passer à la dimension géographique, nous analysons la composition intrinsèque du réseau IRVE national via des représentations graphiques fixes idéales pour l'intégration au rapport final.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7)) # Création d'une figure avec 2 sous-graphiques côte à côte
fig.suptitle("Distribution des bornes IRVE par type d'implantation", fontsize=14, fontweight='bold')

labels = list(COULEURS.keys())
counts = [types_implantation.get(l, 0) for l in labels] 
colors = list(COULEURS.values()) 

# Graphique 1 : Diagramme à barres horizontales
bars = axes[0].barh(labels, counts, color=colors, edgecolor='white')
axes[0].set_xlabel('Nombre de bornes', fontsize=11)
axes[0].set_title('Volume global par catégorie', fontsize=12)
axes[0].invert_yaxis()
axes[0].grid(axis='x', linestyle='--', alpha=0.5)

# Ajout des étiquettes de valeurs sur les barres
for bar in bars:
    width = bar.get_width()
    axes[0].text(width + 200, bar.get_y() + bar.get_height()/2, f'{int(width):,}', 
                 va='center', ha='left', fontsize=10, fontweight='bold')

# Graphique 2 : Diagramme circulaire (Répartition en %)
axes[1].pie(counts, labels=labels, colors=colors, autopct='%1.1f%%', 
            startangle=140, textprops={'fontsize': 10}, wedgeprops={'edgecolor': 'white'})
axes[1].set_title('Proportion du réseau national (%)', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6)) # Création d'une figure avec 2 sous-graphiques côte à côte
fig.suptitle('Analyse distributive de la puissance nominale des bornes IRVE', fontsize=14, fontweight='bold')

# Graphique 1 : Histogramme global du réseau
axes[0].hist(df_bc1['puissance_nominale'].dropna(), bins=50, color='#1976D2', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Puissance nominale (kW)', fontsize=11) #placement de l'axe des abscisses
axes[0].set_ylabel('Nombre de bornes', fontsize=11) #placement de l'axe des ordonnées
axes[0].set_title('Vue d\'ensemble macro (0 à 400+ kW)', fontsize=12) #placement du titre
axes[0].axvline(df_bc1['puissance_nominale'].median(), color='red', linestyle='--', label=f"Médiane : {df_bc1['puissance_nominale'].median()} kW")#placement de la ligne verticale de la médiane
axes[0].legend() #placement de la légende
axes[0].grid(axis='y', linestyle='--', alpha=0.3) #placement de la grille

# Graphique 2 : Zoom sur les bornes standards de recharge lente/accélérée (<= 50 kW)
df_inf_50 = df_bc1[df_bc1['puissance_nominale'] <= 50]
axes[1].hist(df_inf_50['puissance_nominale'].dropna(), bins=25, color='#009688', edgecolor='white', alpha=0.85)
axes[1].set_xlabel('Puissance nominale (kW)', fontsize=11)
axes[1].set_ylabel('Nombre de bornes', fontsize=11)
axes[1].set_title('Focus micro sur la recharge standard (<= 50 kW)', fontsize=12)
axes[1].grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Cartographie Interactive par Type d'Implantation

### Choix méthodologique : L'algorithme MarkerCluster
L'affichage de 70 394 marqueurs individuels provoquerait un crash ou un ralentissement critique du navigateur Web. Pour surmonter cette contrainte de volume, nous implémentons la structure de clustering géospatial de `folium.plugins.MarkerCluster`. Les points se regroupent dynamiquement à grande échelle et se déploient au fil du zoom de l'utilisateur.

In [ ]:
print("Calcul et génération de la carte interactive des implantations...")

# Initialisation de la carte centrée sur le cœur de la France métropolitaine
carte_clustering = folium.Map(location=[46.603354, 1.888334], zoom_start=6, tiles='CartoDB Positron')

# Initialisation de l'instance globale de clustering visuel
cluster_moteur = MarkerCluster(disableClusteringAtZoom=15).add_to(carte_clustering)

# Table de correspondance : Code hexadécimal vers noms de marqueurs natifs Folium
mapping_folium_colors = {
    '#2196F3': 'blue',
    '#4CAF50': 'green',
    '#FF9800': 'orange',
    '#F44336': 'red',
    '#9C27B0': 'purple'
}

# Injection ultra-rapide des 70 394 lignes via des structures zip natives
for lat, lon, imp, hex_c in zip(df_bc1['lat'], df_bc1['lon'], df_bc1['implantation_station'], df_bc1['couleur_hex']):
    folium.CircleMarker(
        location=[lat, lon],
        radius=4,
        color=hex_c,
        fill=True,
        fill_color=hex_c,
        fill_opacity=0.7,
        popup=folium.Popup(f"<b>Station :</b> {imp}<br><b>Position :</b> {lat:.3f}, {lon:.3f}", max_width=300)
    ).add_to(cluster_moteur)

# Sauvegarde physique du livrable HTML
carte_clustering.save('carte_implantations.html')
print("Fichier 'carte_implantations.html' généré avec succès dans le dossier courant.")

# Rendu visuel intégré au Notebook
carte_clustering

## 5. Modélisation de la Densité Spatiale (Heatmaps)

L'analyse macro requiert également la mise en évidence des bassins de concentration d'infrastructures et des disparités territoriales. Nous générons une carte de chaleur globale du réseau, suivie d'une vue sélective dédiée exclusivement aux infrastructures de recharge rapide de forte puissance.

In [ ]:
print("Conception de la carte de chaleur globale...")

# Utilisation d'un fond de plan sombre (Dark Matter) pour faire ressortir le spectre thermique
carte_thermique_globale = folium.Map(location=[46.603354, 1.888334], zoom_start=6, tiles='CartoDB Dark_Matter')

# Extraction matricielle des coordonnées
points_coordonnees = df_bc1[['lat', 'lon']].values.tolist()

# Application de la couche HeatMap à haute densité
HeatMap(
    points_coordonnees,
    radius=7,
    blur=12,
    max_zoom=11,
    gradient={0.2: 'blue', 0.45: 'lime', 0.75: 'orange', 1.0: 'red'}
).add_to(carte_thermique_globale)

# Intégration d'un titre flottant HTML élégant et épuré pour la carte de chaleur globale
titre_html_global = '''
<div style="position: fixed; top: 20px; left: 50%; transform: translateX(-50%);
            z-index:9999; background-color: rgba(30, 30, 30, 0.85); color: white;
            padding: 12px 24px; border-radius: 8px; font-family: sans-serif;
            font-size: 16px; font-weight: bold; box-shadow: 0px 4px 10px rgba(0,0,0,0.5);
            pointer-events: none; text-align: center;">
     Densité Spatiale Globale du Réseau IRVE en France
</div>
'''
carte_thermique_globale.get_root().html.add_child(folium.Element(titre_html_global))

carte_thermique_globale.save('carte_chaleur_densite.html')
print("Fichier 'carte_chaleur_densite.html' généré avec succès.")
carte_thermique_globale

In [ ]:
print("Focus cartographique : Filtrage et génération de la Heatmap de Recharge Rapide...")

# Isolation stricte des bornes à haute performance (Stations dédiées à la recharge rapide)
df_fast_charging = df_bc1[df_bc1['implantation_station'] == 'Station dédiée à la recharge rapide']

carte_thermique_rapide = folium.Map(location=[46.603354, 1.888334], zoom_start=6, tiles='CartoDB Dark_Matter')

points_rapides = df_fast_charging[['lat', 'lon']].values.tolist()

HeatMap(
    points_rapides,
    radius=9,
    blur=14,
    max_zoom=11,
    gradient={0.2: 'navy', 0.5: 'cyan', 0.8: 'yellow', 1.0: 'crimson'}
).add_to(carte_thermique_rapide)

titre_html_rapide = '''
<div style="position: fixed; top: 20px; left: 50%; transform: translateX(-50%);
            z-index:9999; background-color: rgba(40, 10, 10, 0.85); color: #FFCDD2;
            padding: 12px 24px; border-radius: 8px; font-family: sans-serif;
            font-size: 16px; font-weight: bold; box-shadow: 0px 4px 10px rgba(0,0,0,0.5);
            pointer-events: none; text-align: center; border: 1px solid #F44336;">
     Concentration des Stations Dédiées à la Recharge Rapide
</div>
'''
carte_thermique_rapide.get_root().html.add_child(folium.Element(titre_html_rapide))

carte_thermique_rapide.save('carte_chaleur_rapide.html')
print(f"Fichier 'carte_chaleur_rapide.html' généré ({len(df_fast_charging)} stations identifiées).")
carte_thermique_rapide

## 6. Synthèse Méthodologique et Justifications

| Décision Technique | Choix Implémenté | Justification Académique / Professionnelle |
| :--- | :--- | :--- |
| **Variables Isolées** | `lat`, `lon`, `implantation_station`, `puissance_nominale` | Filtrage strict en amont pour maximiser l'efficacité de la RAM et respecter l'isolation fonctionnelle du Besoin 1. |
| **Audit Géospatial** | Vérification des frontières [41, 51.5] Lat et [-5, 10] Lon | Indispensable pour détecter les anomalies de saisie (inversions de coordonnées ou points aberrants) avant l'entraînement des futurs modèles IA. |
| **Représentation Macro** | `folium.plugins.MarkerCluster` | Solution algorithmique incontournable pour intégrer 70 394 observations sur un moteur Leaflet.js sans provoquer l'asphyxie du navigateur. |
| **Analyse de Densité** | `folium.plugins.HeatMap` sur fond *Dark Matter* | Permet de s'affranchir des frontières administratives classiques pour identifier instantanément les zones blanches et les axes majeurs de transit (ex: corridors autoroutiers). |